# 嘉義大學資工系資訊查詢 RAG Chatbot

本專案使用 Retrieval-Augmented Generation（RAG）架構，
從嘉義大學資訊工程學系相關 PDF 文件中檢索資訊，
再透過 Gemini 生成回答，並以 Gradio 建立 Web 問答介面。

### 系統流程

PDF 文件 → 文件切分 → Embedding → Chroma 向量資料庫
→ Retriever → Gemini → Gradio

### 使用技術

- Python
- LangChain
- Gemini 3.5 Flash
- gemini-embedding-001
- Chroma
- PyPDFLoader
- RecursiveCharacterTextSplitter
- Gradio


## 1. 安裝套件

本專案使用 LangChain 處理 RAG 流程，Chroma 儲存向量資料，
PyPDFLoader 載入 PDF，Gradio 建立 Web 介面。


In [1]:
!pip install -q \
    "langchain==0.3.28" \
    "langchain-community==0.3.31" \
    "langchain-text-splitters==0.3.11" \
    "langchain-google-genai" \
    "langchain-chroma" \
    "pypdf" \
    "gradio"


## 2. 匯入套件與設定 API Key

API Key 使用 Google Colab 的 Userdata / Secrets 儲存，
不直接寫入 Notebook。


In [2]:
import os
import hashlib
import shutil

import gradio as gr

from google.colab import userdata
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("API Key 已載入。")

API Key 已載入。


## 3. 設定模型與資料路徑

- LLM：Gemini 3.5 Flash
- Embedding：gemini-embedding-001
- 文件：`csie_.pdf`
- 向量資料庫：Chroma

需先將 `csie_.pdf` 上傳至目前的 Colab 工作目錄。


In [8]:
LLM_MODEL = "gemini-3.5-flash"
EMBEDDING_MODEL = "models/gemini-embedding-001"

PDF_PATH = "./csie_.pdf"
VECTORSTORE_DIR = "./gemini_csie_chroma_vectorstore"

llm = ChatGoogleGenerativeAI(model=LLM_MODEL)
embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

print("模型初始化完成。")


模型初始化完成。


## 4. 建立／更新 Chroma 向量資料庫

為避免每次執行 Notebook 都重新處理 PDF，
先計算 PDF 的 MD5 Hash：

- PDF 未變更 → 載入既有向量資料庫
- PDF 有變更 → 重新載入、切分並建立向量資料庫

這樣可以減少不必要的 Embedding 與資料庫建立工作。


In [4]:
def calculate_file_hash(file_path):
    hasher = hashlib.md5()
    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


def build_or_load_vectorstore(pdf_path, vectorstore_dir):
    hash_file_path = os.path.join(vectorstore_dir, "pdf_hash.txt")
    new_hash = calculate_file_hash(pdf_path)

    # 已存在向量資料庫且 PDF 沒有變更
    if os.path.exists(vectorstore_dir) and os.path.exists(hash_file_path):
        with open(hash_file_path, "r", encoding="utf-8") as file:
            old_hash = file.read().strip()

        if new_hash == old_hash:
            print("PDF 未變更，載入既有 Chroma 向量資料庫。")
            return Chroma(
                persist_directory=vectorstore_dir,
                embedding_function=embeddings
            )

    # PDF 有變更或尚未建立資料庫
    if os.path.exists(vectorstore_dir):
        print("PDF 已變更，刪除舊的向量資料庫。")
        shutil.rmtree(vectorstore_dir)

    print("載入 PDF...")
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    print(f"PDF 載入完成，共 {len(documents)} 頁。")

    print("切分文件...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100
    )
    docs = text_splitter.split_documents(documents)
    print(f"文件切分完成，共 {len(docs)} 個文字片段。")

    print("建立 Chroma 向量資料庫...")
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=vectorstore_dir
    )

    os.makedirs(vectorstore_dir, exist_ok=True)
    with open(hash_file_path, "w", encoding="utf-8") as file:
        file.write(new_hash)

    print("向量資料庫建立完成。")
    return vectorstore


vectorstore = build_or_load_vectorstore(
    PDF_PATH,
    VECTORSTORE_DIR
)


PDF 未變更，載入既有 Chroma 向量資料庫。


## 5. 建立 Retriever 與 RAG 問答鏈

Retriever 會根據使用者問題，從 Chroma 中找出相關文件片段，
再將檢索結果提供給 Gemini 生成回答。


In [9]:
retriever = vectorstore.as_retriever()

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

print("RAG 問答鏈建立完成。")


RAG 問答鏈建立完成。


## 6. 建立 Gradio Web Chatbot

使用者輸入問題後：

1. Gradio 接收問題
2. Retriever 從向量資料庫搜尋相關內容
3. Gemini 根據檢索結果生成回答
4. 將回答顯示於介面


In [10]:
def rag_chain(question):
    result = qa_chain.invoke(question)
    return result["result"]


def get_answer(question):
    if not question or not question.strip():
        return "請輸入想查詢的問題。"

    return rag_chain(question)


with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 嘉義大學資工系 Q&A
        根據提供的嘉義大學資訊工程學系相關文件回答問題。
        """
    )

    question_input = gr.Textbox(
        label="問題",
        placeholder="請輸入你想查詢的資工系相關問題",
        lines=2
    )

    submit_btn = gr.Button("提交")

    answer_output = gr.Textbox(
        label="回答",
        lines=6,
        interactive=False
    )

    submit_btn.click(
        fn=get_answer,
        inputs=question_input,
        outputs=answer_output
    )

    question_input.submit(
        fn=get_answer,
        inputs=question_input,
        outputs=answer_output
    )

print("Gradio 介面建立完成。")


Gradio 介面建立完成。


/tmp/ipykernel_10478/3682381582.py:13: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


## 7. 啟動 Chatbot

執行後會產生 Gradio 的公開連結，可直接開啟 Web Chatbot。


In [11]:
demo.launch(share=True,debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7f740a9853b8d5bd78.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3-flash
Please retry in 13.486188146s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 13
}
].
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash
Please retry in 7h53m55.764660407s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7f740a9853b8d5bd78.gradio.live
